# DNS Tunneling Detection — Data Loading

Load and combine benign and malicious DNS traffic data from CSV files.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. Set Data Paths

In [8]:
data_dir = Path('../data')
benign_dir = data_dir / 'benign-csvs'
malic_dir = data_dir / 'malic-csvs'

print(f'Benign data directory: {benign_dir}')
print(f'Malicious data directory: {malic_dir}')
print(f'Benign files: {list(benign_dir.glob("*.csv"))}')
print(f'Malicious files: {list(malic_dir.glob("*.csv"))}')

Benign data directory: ..\data\benign-csvs
Malicious data directory: ..\data\malic-csvs
Benign files: [WindowsPath('../data/benign-csvs/benign-chrome.csv'), WindowsPath('../data/benign-csvs/benign-firefox.csv')]
Malicious files: [WindowsPath('../data/malic-csvs/mal-dns2tcp.csv'), WindowsPath('../data/malic-csvs/mal-dnscat2.csv'), WindowsPath('../data/malic-csvs/mal-iodine.csv')]


## 2. Load Malicious Data (Small Files)

Malicious CSV files are manageable size, load them directly.

In [16]:
malicious_dfs = {}
for csv_file in malic_dir.glob('*.csv'):
    df = pd.read_csv(csv_file)
    file_name = csv_file.stem
    malicious_dfs[file_name] = df 
    print(f'{file_name}: {df.shape[0]} rows × {df.shape[1]} columns')

print('\nMalicious files loaded successfully.')

mal-dns2tcp: 167517 rows × 35 columns
mal-dnscat2: 35854 rows × 35 columns
mal-iodine: 46598 rows × 35 columns

Malicious files loaded successfully.


## 3. Load Benign Data (Large Files — Using Chunking)

Benign CSV files are >50MB each. Use chunking to load efficiently into memory.

In [10]:
def load_large_csv(filepath, chunksize=10000):
    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunksize):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

benign_dfs = {}
for csv_file in benign_dir.glob('*.csv'):
    print(f'Loading {csv_file.name}... ', end='')
    df = load_large_csv(csv_file, chunksize=10000)
    file_name = csv_file.stem
    benign_dfs[file_name] = df
    print(f'✓ {df.shape[0]} rows × {df.shape[1]} columns')

Loading benign-chrome.csv... ✓ 545464 rows × 35 columns
Loading benign-firefox.csv... ✓ 371836 rows × 35 columns


## 4. Inspect Column Names and Data Types

In [ ]:
first_mal = list(malicious_dfs.values())[0]
print('Columns in malicious files:')
print(first_mal.columns.tolist())
print(f'\nData types:\n{first_mal.dtypes}')

Columns in malicious files:
['SourceIP', 'DestinationIP', 'SourcePort', 'DestinationPort', 'TimeStamp', 'Duration', 'FlowBytesSent', 'FlowSentRate', 'FlowBytesReceived', 'FlowReceivedRate', 'PacketLengthVariance', 'PacketLengthStandardDeviation', 'PacketLengthMean', 'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian', 'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation', 'PacketTimeVariance', 'PacketTimeStandardDeviation', 'PacketTimeMean', 'PacketTimeMedian', 'PacketTimeMode', 'PacketTimeSkewFromMedian', 'PacketTimeSkewFromMode', 'PacketTimeCoefficientofVariation', 'ResponseTimeTimeVariance', 'ResponseTimeTimeStandardDeviation', 'ResponseTimeTimeMean', 'ResponseTimeTimeMedian', 'ResponseTimeTimeMode', 'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode', 'ResponseTimeTimeCoefficientofVariation', 'DoH']

Data types:
SourceIP                                   object
DestinationIP                              object
SourcePort                   

In [18]:
print(f'\nFirst row:')
print(first_mal.head(1))


First row:
         SourceIP DestinationIP  SourcePort  DestinationPort  \
0  192.168.20.209       1.1.1.1       39406              443   

             TimeStamp    Duration  FlowBytesSent  FlowSentRate  \
0  2020-04-01 22:55:13  120.772871          42357     350.71618   

   FlowBytesReceived  FlowReceivedRate  ...  PacketTimeCoefficientofVariation  \
0              71915        595.456574  ...                          0.489724   

   ResponseTimeTimeVariance  ResponseTimeTimeStandardDeviation  \
0                   0.40235                           0.634311   

   ResponseTimeTimeMean  ResponseTimeTimeMedian  ResponseTimeTimeMode  \
0              0.163861                0.001734              0.000006   

   ResponseTimeTimeSkewFromMedian  ResponseTimeTimeSkewFromMode  \
0                        0.766785                      0.258319   

   ResponseTimeTimeCoefficientofVariation   DoH  
0                                3.871039  True  

[1 rows x 35 columns]


## 5. Label Classes

In [22]:
for key in benign_dfs:
    print(f'Adding Label column to {key}...')
    benign_dfs[key]['Label'] = 0
for key in malicious_dfs:
    print(f'Adding Label column to {key}...')
    malicious_dfs[key]['Label'] = 1

print('Labels added: Benign=0, Malicious=1')

Adding Label column to benign-chrome...
Adding Label column to benign-firefox...
Adding Label column to mal-dns2tcp...
Adding Label column to mal-dnscat2...
Adding Label column to mal-iodine...
Labels added: Benign=0, Malicious=1


## 6. Combine All Data

In [23]:
all_dfs = list(benign_dfs.values()) + list(malicious_dfs.values())
df_combined = pd.concat(all_dfs, ignore_index=True)

print(f'Combined Dataset Shape: {df_combined.shape}')
print(f'Rows: {df_combined.shape[0]:,}')
print(f'Columns: {df_combined.shape[1]}')

Combined Dataset Shape: (1167269, 36)
Rows: 1,167,269
Columns: 36


## 7. Basic Validation

In [13]:
print('=== DATA VALIDATION ===')
print(f'Missing values: {df_combined.isnull().sum().sum()}')
print(f'Duplicate rows: {df_combined.duplicated().sum()}')
print(f'\nLabel distribution:\n{df_combined["Label"].value_counts()}')

=== DATA VALIDATION ===
Missing values: 16056
Duplicate rows: 0

Label distribution:
Label
0    917300
1    249969
Name: count, dtype: int64


## 8. Summary Statistics

In [24]:
print('=== SUMMARY STATISTICS ===')
print(f'Total rows: {df_combined.shape[0]:,}')
print(f'Total columns: {df_combined.shape[1]}')
for label, count in df_combined['Label'].value_counts().items():
    label_name = 'Benign' if label == 0 else 'Malicious'
    percentage = (count / len(df_combined)) * 100
    print(f'{label_name}: {count:,} ({percentage:.2f}%)')

=== SUMMARY STATISTICS ===
Total rows: 1,167,269
Total columns: 36
Benign: 917,300 (78.59%)
Malicious: 249,969 (21.41%)


## 9. Save Combined Dataset

In [25]:
output_file = Path('../data/combined_dns_data.csv')
df_combined.to_csv(output_file, index=False)
print(f'✓ Saved to {output_file}')
print(f'Rows: {len(df_combined):,}')
print(f'Columns: {len(df_combined.columns)}')

✓ Saved to ..\data\combined_dns_data.csv
Rows: 1,167,269
Columns: 36
